In [18]:
team_matches.to_csv("../data/features/match_features.csv", index=False)
team_matches.shape

(760, 22)

In [17]:
import os
os.makedirs("../data/features", exist_ok=True)

In [16]:
team_matches = team_matches.merge(
    h2h_df[["team", "utcDate", "h2h_win_pct"]],
    on=["team", "utcDate"],
    how="left"
)

In [15]:
h2h_df[h2h_df["h2h_win_pct"].notna()].head(10)

,team,opponent,utcDate,goals_for,goals_against,h2h_win,h2h_win_pct
1,AFC Bournemouth,Arsenal FC,2026-04-11 11:30:00+00:00,2,1,1,0.0
3,AFC Bournemouth,Aston Villa FC,2026-02-07 15:00:00+00:00,1,1,0,0.0
5,AFC Bournemouth,Brentford FC,2026-03-03 19:30:00+00:00,0,0,0,0.0
7,AFC Bournemouth,Brighton & Hove Albion FC,2026-01-19 20:00:00+00:00,1,1,0,1.0
9,AFC Bournemouth,Burnley FC,2026-03-14 15:00:00+00:00,0,0,0,0.0
11,AFC Bournemouth,Chelsea FC,2025-12-30 19:30:00+00:00,2,2,0,0.0
13,AFC Bournemouth,Crystal Palace FC,2026-05-03 13:00:00+00:00,3,0,1,0.0
15,AFC Bournemouth,Everton FC,2026-02-10 19:30:00+00:00,2,1,1,0.0
17,AFC Bournemouth,Fulham FC,2026-05-09 14:00:00+00:00,1,0,1,1.0
19,AFC Bournemouth,Leeds United FC,2026-04-22 19:00:00+00:00,2,2,0,0.0


In [14]:
h2h_df["h2h_win"] = (h2h_df["goals_for"] > h2h_df["goals_against"]).astype(int)

h2h_df["h2h_win_pct"] = (
    h2h_df.groupby(["team", "opponent"])["h2h_win"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [13]:
h2h_records = []

for _, match in clean_matches.iterrows():
    h2h_records.append({"team": match["home_team_name"], "opponent": match["away_team_name"],
                         "utcDate": match["utcDate"], "goals_for": match["home_goals"], "goals_against": match["away_goals"]})
    h2h_records.append({"team": match["away_team_name"], "opponent": match["home_team_name"],
                         "utcDate": match["utcDate"], "goals_for": match["away_goals"], "goals_against": match["home_goals"]})

h2h_df = pd.DataFrame(h2h_records).sort_values(["team", "opponent", "utcDate"]).reset_index(drop=True)

In [12]:
team_matches[team_matches["team"] == "AFC Bournemouth"][
    ["utcDate", "is_home", "goal_difference", "clean_sheet_pct", "days_since_last_match", "home_win_pct", "away_win_pct"]
].head(10)

,utcDate,is_home,goal_difference,clean_sheet_pct,days_since_last_match,home_win_pct,away_win_pct
0,2025-08-15 19:00:00+00:00,False,-2,NaN,NaN,NaN,NaN
1,2025-08-23 14:00:00+00:00,True,1,0.000000,7.0,NaN,NaN
2,2025-08-30 14:00:00+00:00,False,1,0.500000,7.0,NaN,0.000000
3,2025-09-13 14:00:00+00:00,True,1,0.666667,14.0,1.000000,NaN
4,2025-09-21 13:00:00+00:00,True,0,0.500000,7.0,1.000000,NaN
5,2025-09-27 14:00:00+00:00,False,0,0.600000,6.0,NaN,0.500000
6,2025-10-03 19:00:00+00:00,True,2,0.500000,6.0,0.666667,NaN
7,2025-10-18 14:00:00+00:00,False,0,0.428571,14.0,NaN,0.333333
8,2025-10-26 14:00:00+00:00,True,2,0.375000,8.0,0.750000,NaN
9,2025-11-02 16:30:00+00:00,False,-2,0.444444,7.0,NaN,0.250000


In [11]:
team_matches["is_win"] = (team_matches["result"] == "WIN").astype(int)

team_matches["home_win_pct"] = (
    team_matches[team_matches["is_home"]]
    .groupby("team")["is_win"]
    .transform(lambda x: x.shift(1).expanding().mean())
)
team_matches["away_win_pct"] = (
    team_matches[~team_matches["is_home"]]
    .groupby("team")["is_win"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [10]:
team_matches["days_since_last_match"] = (
    team_matches.groupby("team")["utcDate"]
    .diff()
    .dt.days
)

In [9]:
team_matches["clean_sheet"] = (team_matches["goals_conceded"] == 0).astype(int)
team_matches["clean_sheet_pct"] = (
    team_matches.groupby("team")["clean_sheet"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [8]:
team_matches["goal_difference"] = team_matches["goals_scored"] - team_matches["goals_conceded"]

In [7]:
team_matches[team_matches["team"] == "AFC Bournemouth"][
    ["utcDate", "result", "points", "rolling_form_pts", "win_streak", "loss_streak", "avg_goals_scored", "avg_goals_conceded"]
].head(10)

,utcDate,result,points,rolling_form_pts,win_streak,loss_streak,avg_goals_scored,avg_goals_conceded
0,2025-08-15 19:00:00+00:00,LOSS,0,NaN,0,0,NaN,NaN
1,2025-08-23 14:00:00+00:00,WIN,3,0.0,0,1,2.000000,4.000000
2,2025-08-30 14:00:00+00:00,WIN,3,3.0,1,0,1.500000,2.000000
3,2025-09-13 14:00:00+00:00,WIN,3,6.0,2,0,1.333333,1.333333
4,2025-09-21 13:00:00+00:00,DRAW,1,9.0,3,0,1.500000,1.250000
5,2025-09-27 14:00:00+00:00,DRAW,1,10.0,0,0,1.200000,1.000000
6,2025-10-03 19:00:00+00:00,WIN,3,11.0,0,0,1.333333,1.166667
7,2025-10-18 14:00:00+00:00,DRAW,1,11.0,1,0,1.571429,1.142857
8,2025-10-26 14:00:00+00:00,WIN,3,9.0,0,0,1.750000,1.375000
9,2025-11-02 16:30:00+00:00,LOSS,0,9.0,1,0,1.777778,1.222222


In [6]:
team_matches["avg_goals_scored"] = (
    team_matches.groupby("team")["goals_scored"]
    .transform(lambda x: x.shift(1).expanding().mean())
)
team_matches["avg_goals_conceded"] = (
    team_matches.groupby("team")["goals_conceded"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [5]:
def compute_streak(results, target):
    streak = 0
    streaks = []
    for r in results:
        streaks.append(streak)  # streak BEFORE this match
        streak = streak + 1 if r == target else 0
    return streaks

team_matches["win_streak"] = team_matches.groupby("team")["result"].transform(
    lambda x: compute_streak(x, "WIN")
)
team_matches["loss_streak"] = team_matches.groupby("team")["result"].transform(
    lambda x: compute_streak(x, "LOSS")
)

In [4]:
team_matches["rolling_form_pts"] = (
    team_matches.groupby("team")["points"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).sum())
)

In [3]:
points_map = {"WIN": 3, "DRAW": 1, "LOSS": 0}
team_matches["points"] = team_matches["result"].map(points_map)

In [2]:
home_rows = clean_matches[["utcDate", "matchday", "id", "home_team_name", "home_goals", "away_goals", "winner"]].copy()
home_rows.columns = ["utcDate", "matchday", "match_id", "team", "goals_scored", "goals_conceded", "winner"]
home_rows["is_home"] = True
home_rows["result"] = home_rows["winner"].map({"HOME_TEAM": "WIN", "AWAY_TEAM": "LOSS", "DRAW": "DRAW"})

away_rows = clean_matches[["utcDate", "matchday", "id", "away_team_name", "away_goals", "home_goals", "winner"]].copy()
away_rows.columns = ["utcDate", "matchday", "match_id", "team", "goals_scored", "goals_conceded", "winner"]
away_rows["is_home"] = False
away_rows["result"] = away_rows["winner"].map({"HOME_TEAM": "LOSS", "AWAY_TEAM": "WIN", "DRAW": "DRAW"})

team_matches = pd.concat([home_rows, away_rows]).sort_values(["team", "utcDate"]).reset_index(drop=True)
team_matches.drop(columns="winner", inplace=True)

In [1]:
import pandas as pd

clean_matches = pd.read_csv("../data/processed/clean_matches.csv", parse_dates=["utcDate"])
clean_matches = clean_matches.sort_values("utcDate").reset_index(drop=True)